# 03. Model Coxa i diagnostyka semiparametryczna

            Notebook odtwarza model Coxa, test proporcjonalności hazardów, reszty Schoenfelda, wariant z efektem zależnym od czasu, obserwacje wpływowe oraz predykcje przeżycia dla profili pacjentów.


## Import bibliotek i przygotowanie danych


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# UWAGA:
# Plik z danymi powinien znajdować się najlepiej w tym samym folderze co notebook
# albo w podfolderze "data".
#
# Oczekiwana nazwa pliku:
# heart_failure_clinical_records_dataset.csv
#
# Kod nie używa bezwzględnych ścieżek Windows, dzięki czemu notebook powinien
# działać po przeniesieniu do dowolnego katalogu roboczego.

DATA_FILENAME = "heart_failure_clinical_records_dataset.csv"


def find_data_file(filename: str = DATA_FILENAME) -> Path:
    """
    Szuka pliku danych w typowych lokalizacjach względem katalogu roboczego
    i jego katalogów nadrzędnych.

    Priorytet:
    1. bieżący katalog roboczy,
    2. podfolder data,
    3. podfolder Data,
    4. katalogi nadrzędne,
    5. rekurencyjne wyszukiwanie w bieżącym katalogu i katalogach nadrzędnych.
    """
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]

    direct_candidates = []
    for root in search_roots:
        direct_candidates.extend([
            root / filename,
            root / "data" / filename,
            root / "Data" / filename,
        ])

    for candidate in direct_candidates:
        if candidate.exists():
            return candidate.resolve()

    for root in search_roots:
        matches = list(root.rglob(filename))
        if matches:
            return matches[0].resolve()

    raise FileNotFoundError(
        f"Nie znaleziono pliku danych: {filename}\n"
        "Umieść plik w tym samym folderze co notebook albo w podfolderze 'data'."
    )


DATA_PATH = find_data_file()

print(f"Wczytywany plik danych: {DATA_PATH}")

df = pd.read_csv(DATA_PATH).drop_duplicates().copy()

df["duration"] = df["time"].astype(float)
df["event"] = df["DEATH_EVENT"].astype(int)

df["age_60plus"] = (df["age"] >= 60).astype(int)
df["ef_low"] = (df["ejection_fraction"] < 35).astype(int)
df["creatinine_high"] = (df["serum_creatinine"] > 1.5).astype(int)
df["sodium_low"] = (df["serum_sodium"] < 135).astype(int)

df["log_cpk"] = np.log1p(df["creatinine_phosphokinase"])
df["log_platelets"] = np.log(df["platelets"])


def show(title, obj):
    print(f"\n{title}")
    print("-" * len(title))
    if isinstance(obj, pd.DataFrame):
        print(obj.to_string())
    elif isinstance(obj, pd.Series):
        print(obj.to_string())
    else:
        print(obj)


from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from scipy.stats import chi2

cox_vars = [
    "age",
    "ejection_fraction",
    "serum_creatinine",
    "serum_sodium",
    "anaemia",
    "diabetes",
    "high_blood_pressure",
    "sex",
    "smoking",
    "log_cpk",
]

cox_df = df[["duration", "event"] + cox_vars].copy()

show("Zmiennie modelu Coxa", pd.Series(cox_vars))

## Model Coxa


In [ ]:
cph = CoxPHFitter(penalizer=0.05)
cph.fit(cox_df, duration_col="duration", event_col="event")
cox_summary = cph.summary[["coef", "exp(coef)", "se(coef)", "p", "coef lower 95%", "coef upper 95%"]].round(4)
show("Wyniki modelu Coxa", cox_summary)

fig, ax = plt.subplots(figsize=(7.4, 5.2))
coefs = cox_summary["coef"]
lower = cox_summary["coef lower 95%"]
upper = cox_summary["coef upper 95%"]
y = np.arange(len(coefs))
ax.errorbar(coefs, y, xerr=[coefs - lower, upper - coefs], fmt="o", color="#4C72B0", ecolor="#4C72B0", capsize=3)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(coefs.index)
ax.set(title="Współczynniki modelu Coxa", xlabel="log hazard ratio")
plt.tight_layout()


## Test proporcjonalności hazardów


In [ ]:
ph = proportional_hazard_test(cph, cox_df, time_transform="rank")
ph_table = ph.summary[["test_statistic", "p"]].sort_values("p").round(4)
show("Test proporcjonalności hazardów", ph_table)


## Reszty Schoenfelda


In [ ]:
schoenfeld = cph.compute_residuals(cox_df, kind="schoenfeld")
event_times = cox_df.loc[schoenfeld.index, "duration"]
show("Fragment macierzy reszt Schoenfelda", schoenfeld.head().round(4))

plot_vars = cox_vars[:5]
fig, axes = plt.subplots(len(plot_vars), 1, figsize=(8, 10), sharex=True)
for ax, var in zip(axes, plot_vars):
    ax.scatter(event_times, schoenfeld[var], s=14, alpha=0.65, color="#4C72B0")
    z = np.polyfit(event_times, schoenfeld[var], 1)
    ax.plot(np.sort(event_times), np.polyval(z, np.sort(event_times)), color="#C44E52", lw=1.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(var)
axes[-1].set_xlabel("Czas zdarzenia")
fig.suptitle("Reszty Schoenfelda - model podstawowy", y=1.01)
plt.tight_layout()

remaining = cox_vars[5:]
fig, axes = plt.subplots(len(remaining), 1, figsize=(8, 10), sharex=True)
for ax, var in zip(axes, remaining):
    ax.scatter(event_times, schoenfeld[var], s=14, alpha=0.65, color="#4C72B0")
    z = np.polyfit(event_times, schoenfeld[var], 1)
    ax.plot(np.sort(event_times), np.polyval(z, np.sort(event_times)), color="#C44E52", lw=1.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(var)
axes[-1].set_xlabel("Czas zdarzenia")
fig.suptitle("Reszty Schoenfelda - pozostałe zmienne", y=1.01)
plt.tight_layout()


## Model Coxa z interakcją czasową dla frakcji wyrzutowej


In [ ]:
from lifelines import CoxTimeVaryingFitter

# Rozszerzony model Coxa z efektem frakcji wyrzutowej zależnym od czasu.
# Uwaga: nie tworzymy EF * log(duration) jako zwykłej stałej zmiennej.
# Zamiast tego rozwijamy dane do formatu start-stop, aby składnik
# EF * log(1 + t) był liczony względem aktualnego czasu ryzyka.

def make_start_stop_data(
    base_df: pd.DataFrame,
    covariates: list[str],
    duration_col: str = "duration",
    event_col: str = "event",
    id_col: str = "id"
) -> pd.DataFrame:
    """
    Tworzy dane w formacie start-stop dla CoxTimeVaryingFitter.

    Każdy pacjent jest dzielony na przedziały kończące się w czasach zdarzeń.
    Dzięki temu składnik ejection_fraction * log(1 + t) może zmieniać się
    w czasie ryzyka.

    Wynikowe kolumny:
    - id: identyfikator pacjenta,
    - start: początek przedziału,
    - stop: koniec przedziału,
    - event: 1 tylko w ostatnim przedziale pacjenta, jeśli zdarzenie wystąpiło,
    - covariates: stałe zmienne objaśniające,
    - ef_log_time: składnik czasowy EF * log(1 + stop).
    """
    data = base_df[[duration_col, event_col] + covariates].copy().reset_index(drop=True)
    data[id_col] = np.arange(len(data))

    # Czasy zdarzeń wyznaczają momenty, w których aktualizujemy zbiory ryzyka.
    event_times = np.sort(data.loc[data[event_col] == 1, duration_col].unique())

    rows = []

    for _, row in data.iterrows():
        subject_id = int(row[id_col])
        duration = float(row[duration_col])
        event = int(row[event_col])

        # Wszystkie czasy zdarzeń wcześniejsze niż końcowy czas pacjenta
        # plus końcowy czas obserwacji pacjenta.
        internal_cuts = event_times[event_times < duration]
        stop_times = np.r_[internal_cuts, duration]

        start = 0.0

        for stop in stop_times:
            stop = float(stop)

            if stop <= start:
                continue

            new_row = {
                id_col: subject_id,
                "start": start,
                "stop": stop,
                event_col: int(event == 1 and np.isclose(stop, duration)),
            }

            for var in covariates:
                new_row[var] = row[var]

            # Właściwa interakcja z aktualnym czasem ryzyka:
            # dla przedziału (start, stop] używamy log(1 + stop).
            new_row["ef_log_time"] = row["ejection_fraction"] * np.log1p(stop)

            rows.append(new_row)
            start = stop

    return pd.DataFrame(rows)


# Zmienne bazowe modelu Coxa.
cox_td_vars_base = cox_vars.copy()

# Dane w formacie start-stop.
cox_tv = make_start_stop_data(
    base_df=cox_df,
    covariates=cox_td_vars_base,
    duration_col="duration",
    event_col="event",
    id_col="id"
)

cox_tv_vars = cox_td_vars_base + ["ef_log_time"]

show(
    "Struktura danych start-stop",
    pd.Series({
        "liczba_pacjentów": cox_df.shape[0],
        "liczba_wierszy_start_stop": cox_tv.shape[0],
        "liczba_zdarzeń": int(cox_tv["event"].sum()),
    })
)

# Estymacja rozszerzonego modelu Coxa.
cph_td = CoxTimeVaryingFitter(penalizer=0.01)

cph_td.fit(
    cox_tv[["id", "start", "stop", "event"] + cox_tv_vars],
    id_col="id",
    start_col="start",
    stop_col="stop",
    event_col="event"
)

td_summary = cph_td.summary.loc[
    ["ejection_fraction", "ef_log_time"],
    ["coef", "exp(coef)", "se(coef)", "p"]
].round(4)

td_summary2 = cph_td.summary.round(4)

show("Model Coxa z efektem frakcji wyrzutowej zależnym od czasu", td_summary)
show("Pełne wyniki modelu Coxa z efektem zależnym od czasu", td_summary2)


# Porównanie z modelem podstawowym.
# Używamy częściowej log-wiarygodności i częściowego AIC,
# ponieważ modele Coxa są estymowane przez częściową wiarygodność.
def partial_aic(model) -> float:
    return -2 * model.log_likelihood_ + 2 * len(model.params_)


compare = pd.DataFrame({
    "model": ["Cox podstawowy", "Cox z EF × log(1+t)"],
    "logLik": [cph.log_likelihood_, cph_td.log_likelihood_],
    "partial_AIC": [partial_aic(cph), partial_aic(cph_td)],
    "liczba_parametrów": [len(cph.params_), len(cph_td.params_)],
}).round(3)

show("Porównanie modelu podstawowego i modelu z efektem zależnym od czasu", compare)


# Interpretacja czasowo zmiennego efektu frakcji wyrzutowej.
# Efekt EF w czasie:
# beta_EF(t) = beta_EF + beta_EF,t * log(1 + t)
# HR_EF(t)   = exp(beta_EF(t))

beta_ef = cph_td.params_["ejection_fraction"]
beta_ef_time = cph_td.params_["ef_log_time"]

time_points = np.array([1, 7, 14, 30, 60, 100, 180, 250], dtype=float)

ef_effect_time = pd.DataFrame({
    "czas_dni": time_points,
    "coef_EF(t)": beta_ef + beta_ef_time * np.log1p(time_points),
})

ef_effect_time["HR_EF(t)"] = np.exp(ef_effect_time["coef_EF(t)"])

show(
    "Czasowo zmienny efekt frakcji wyrzutowej",
    ef_effect_time.round(4)
)

In [ ]:
from lifelines.utils import concordance_index
import numpy as np
import pandas as pd

def partial_aic(model):
    """
    Częściowe AIC dla modelu Coxa:
    AIC_partial = -2 * partial log-likelihood + 2 * liczba parametrów.
    """
    return -2 * model.log_likelihood_ + 2 * len(model.params_)


def cindex_basic_cox(cph, df, duration_col="duration", event_col="event"):
    """
    C-index dla klasycznego modelu Coxa.
    W lifelines większy partial hazard oznacza większe ryzyko,
    a concordance_index oczekuje większej wartości dla dłuższego przeżycia,
    dlatego używamy znaku minus.
    """
    risk_score = cph.predict_partial_hazard(df).values.ravel()
    return concordance_index(
        df[duration_col],
        -risk_score,
        df[event_col]
    )


def cindex_timevarying_at_t(cph_td, df, covariates, t_ref=100,
                            duration_col="duration", event_col="event"):
    """
    Przybliżony apparent C-index dla modelu CoxTimeVaryingFitter
    w ustalonym punkcie czasu t_ref.

    Dla każdego pacjenta tworzymy liniowy predyktor:
    eta_i(t) = beta^T X_i + beta_EF_time * EF_i * log(1 + t_ref)

    To NIE jest pełna walidacja dynamiczna modelu time-varying,
    ale pozwala uzyskać porównywalną, orientacyjną miarę dyskryminacji.
    """
    params = cph_td.params_.copy()

    X_static = df[covariates].copy()
    eta = np.zeros(len(df))

    for var in covariates:
        if var in params.index:
            eta += params[var] * X_static[var].values

    if "ef_log_time" in params.index:
        eta += params["ef_log_time"] * df["ejection_fraction"].values * np.log1p(t_ref)

    return concordance_index(
        df[duration_col],
        -eta,
        df[event_col]
    )


# Punkt czasu dla orientacyjnego C-index modelu time-varying.

t_ref = 100

compare_new = pd.DataFrame({
    "Model": [
        "Cox podstawowy",
        rf"Cox z EF $\times \log(1+t)$, $t={t_ref}$"
    ],
    "logLik": [
        cph.log_likelihood_,
        cph_td.log_likelihood_
    ],
    "AIC częściowy": [
        partial_aic(cph),
        partial_aic(cph_td)
    ],
    "Param.": [
        len(cph.params_),
        len(cph_td.params_)
    ],
    "C-index": [
        cindex_basic_cox(cph, cox_df),
        cindex_timevarying_at_t(cph_td, cox_df, cox_vars, t_ref=t_ref)
    ]
}).round({
    "logLik": 2,
    "AIC częściowy": 2,
    "C-index": 3
})

show("Porównanie modelu Coxa podstawowego i rozszerzonego", compare_new)
compare_new

## Predykcje funkcji przeżycia z modelu Coxa


In [ ]:
profiles = pd.DataFrame({
    "age": [50, 65, 75],
    "ejection_fraction": [45, 35, 20],
    "serum_creatinine": [1.0, 1.4, 2.2],
    "serum_sodium": [140, 137, 132],
    "anaemia": [0, 0, 1],
    "diabetes": [0, 1, 1],
    "high_blood_pressure": [0, 1, 1],
    "sex": [0, 1, 1],
    "smoking": [0, 0, 1],
    "log_cpk": [df["log_cpk"].quantile(0.25), df["log_cpk"].median(), df["log_cpk"].quantile(0.75)],
}, index=["Niskie ryzyko", "Średnie ryzyko", "Wysokie ryzyko"])[cox_vars]
surv = cph.predict_survival_function(profiles, times=np.linspace(1, df["duration"].max(), 250))
fig, ax = plt.subplots(figsize=(7.6, 4.8))
for label, color in zip(profiles.index, ["#55A868", "#DD8452", "#C44E52"]):
    ax.plot(surv.index, surv[label], lw=2.2, label=label, color=color)
ax.set(title="Predykcja funkcji przeżycia z modelu Coxa", xlabel="Czas (dni)", ylabel="S(t)", ylim=(0, 1.02))
ax.legend()
plt.tight_layout()


In [ ]:
# Predykcja funkcji hazardu z modelu Coxa dla trzech profili ryzyka.

baseline_hazard = cph.baseline_hazard_.iloc[:, 0]
partial_hazards = cph.predict_partial_hazard(profiles)
hazard_profiles = pd.DataFrame(index=baseline_hazard.index)
for label in profiles.index:
    hazard_profiles[label] = baseline_hazard * float(partial_hazards.loc[label])

hazard_plot = hazard_profiles.rolling(window=5, min_periods=1, center=True).mean()
fig, ax = plt.subplots(figsize=(9, 5.4))
for label, color in zip(profiles.index, ["#55A868", "#DD8452", "#C44E52"]):
    ax.plot(hazard_plot.index, hazard_plot[label], lw=2.2, label=label, color=color)
ax.set(
    title="Model Coxa: predykcja funkcji hazardu dla klas ryzyka",
    xlabel="Czas (dni)",
    ylabel="h(t)",
)
ax.grid(True, color="#dddddd", linewidth=0.8, alpha=0.8)
ax.legend(title="Profil pacjenta")
fig.tight_layout()
plt.show()
show("Predykcja hazardu z modelu Coxa: pierwsze warto?ci", hazard_profiles.head().round(6))


In [ ]:
# Predykcja skumulowanego hazardu z modelu Coxa dla trzech profili ryzyka.

baseline_cumhaz = cph.baseline_cumulative_hazard_.iloc[:, 0]

partial_hazards = cph.predict_partial_hazard(profiles)

cumhaz_profiles = pd.DataFrame(index=baseline_cumhaz.index)

for label in profiles.index:
    cumhaz_profiles[label] = baseline_cumhaz * float(partial_hazards.loc[label])

fig, ax = plt.subplots(figsize=(9, 5.4))

for label, color in zip(profiles.index, ["#55A868", "#DD8452", "#C44E52"]):
    ax.plot(
        cumhaz_profiles.index,
        cumhaz_profiles[label],
        lw=2.2,
        label=label,
        color=color
    )

ax.set(
    title="Model Coxa: predykcja skumulowanego hazardu dla klas ryzyka",
    xlabel="Czas (dni)",
    ylabel=r"$H(t)$",
)

ax.grid(True, color="#dddddd", linewidth=0.8, alpha=0.8)
ax.legend(title="Profil pacjenta")

fig.tight_layout()
plt.show()

show(
    "Predykcja skumulowanego hazardu z modelu Coxa: pierwsze wartości",
    cumhaz_profiles.head().round(6)
)

## Wnioski kontrolne

            Notebook odtwarza wyniki semiparametryczne użyte w prezentacji i pokazuje sposób obliczenia diagnostyki PH, reszt Schoenfelda, obserwacji wpływowych oraz predykcji profili.
